# d2oracle — Active-Patch Two-Stage Draft Model Training & ONNX Export

Trains the unified **Immortal Ranked + Pro** two-stage Draft Transformer on the active-patch Hugging Face dataset (`rizeru/d2oracle-data`) using a Colab T4 GPU and exports float32 ONNX models for 100% client-side browser inference (`onnxruntime-web`).

**Stage 1 (Match-level outcome & early-game context from 10 heroes + positions 1–5):**
1. Radiant vs Dire win probability
2. Match duration quantiles (`p25 / p50 / p75` minutes)
3. Lane outcomes (Top / Mid / Bot: Radiant win / Even / Dire win) + `@10m` Radiant gold & XP leads

**Stage 2 (Scenario-conditioned per-hero outputs + `duration_override` support):**
4. Key item progression sequence & expected purchase minutes
5. Ability skill order (levels 1–18 with legal hero masks) + talents (10/15/20/25)
6. Combat efficiency rates (`hero_dpm`, `tower_dpm`, `damage_taken_dpm`, `hero_hpm`) & team shares `%`
7. Economy & KDA quantile ranges (`GPM`, `XPM`, `CS/min`, `K/D/A per 10m`)

> Security note: `rizeru/d2oracle-data` is public. Never hardcode tokens inside this notebook; pass `HF_TOKEN` via Colab Secrets only if uploading exported weights.

In [ ]:
# 1. Setup repository & Python dependencies (preserves Colab CUDA PyTorch)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/ElRizeru/d2oracle.git'
if (Path.cwd() / 'training').exists() and (Path.cwd() / 'shared').exists():
    REPO_ROOT = Path.cwd()
elif Path('/content/d2oracle/training').exists():
    REPO_ROOT = Path('/content/d2oracle')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    REPO_ROOT = Path('/content/d2oracle')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'huggingface-hub>=0.26.0', 'pyarrow>=17.0.0', 'onnx>=1.16.0', 'onnxruntime>=1.19.0', 'matplotlib>=3.9.0',
], check=True)
print('Repo ready:', REPO_ROOT)

In [ ]:
# 2. Training configuration, persistent Drive storage (if available), & GPU check
import logging
import sys
from pathlib import Path

import torch

from training.config import MODEL_SCHEMA_VERSION, TrainingConfig

print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| Model schema v:', MODEL_SCHEMA_VERSION)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

USE_GOOGLE_DRIVE = True
ARTIFACT_ROOT = REPO_ROOT
if USE_GOOGLE_DRIVE and 'google.colab' in sys.modules:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_root = Path('/content/drive/MyDrive/d2oracle')
        drive_root.mkdir(parents=True, exist_ok=True)
        ARTIFACT_ROOT = drive_root
    except Exception as exc:
        print('Google Drive unavailable, falling back to local storage:', exc)
print('Artifact root:', ARTIFACT_ROOT)

cfg = TrainingConfig(
    hf_repo_id='rizeru/d2oracle-data',
    data_dir=ARTIFACT_ROOT / 'data/training',
    checkpoint_dir=ARTIFACT_ROOT / 'checkpoints',
    export_dir=ARTIFACT_ROOT / 'exports',
    d_model=192,
    nhead=6,
    num_encoder_layers=4,
    num_decoder_layers=2,
    dim_feedforward=512,
    dropout=0.1,
    max_item_len=12,
    stage1_pair_dim=24,
    stage1_pair_dropout=0.25,
    use_stage2_block=True,
    eos_logit_penalty=0.0,
    eos_stop_prob=0.48,
    batch_size=256 if torch.cuda.is_available() else 32,
    epochs=25,
    two_phase=True,
    stage1_epochs=8,
    lr=4.5e-4 if torch.cuda.is_available() else 3e-4,
    weight_decay=2e-2,
    val_split=0.1,
    pro_val_min_count=100,
    scenario_teacher_forcing_start=0.5,
    scenario_teacher_forcing_end=0.15,
    duration_teacher_forcing_prob=0.40,
    loser_build_weight=0.35,
    win_label_smoothing=0.02,
    w_win=2.0,
    w_duration=2.0,
    w_item_time=1.5,
    resume=True,
    num_workers=0,
    use_amp=torch.cuda.is_available(),
)
print(cfg)

In [ ]:
# 3. Download active-patch v2 Parquet shards & vocab.json from Hugging Face (zero RAM row caching)
import os

from training.dataset import VocabIndex, download_hf_dataset


def _resolve_hf_token() -> str | None:
    tok = os.environ.get('HF_TOKEN')
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN') or None
    except Exception:
        return None

hf_token = _resolve_hf_token()
manifest, vocab, shard_paths = download_hf_dataset(cfg.hf_repo_id, cfg.data_dir, token=hf_token)
vidx = VocabIndex(vocab)
print(f"Active patch: {manifest['game_version_name']} (gv={manifest['game_version_id']}) | schema={manifest.get('parquet_schema_version')}")
print(f"Shards: {len(shard_paths)} | Declared matches: {manifest.get('total_matches')}")
print(f"Heroes: {vidx.num_heroes} | Item vocab: {vidx.item_vocab_size} | Max hero skills: {vidx.max_hero_skills}")

In [ ]:
# 4. Stream shards into compact tensors & train unified two-stage Draft Transformer
from training.train import train_from_shards

train_result = train_from_shards(shard_paths, vocab, cfg, manifest=manifest)
print('Best checkpoint:', train_result['best_checkpoint'])
print('Last val metrics:', train_result['history'][-1]['val'])

In [ ]:
# 5. Plot Stage-1 & Stage-2 training and validation metrics
import matplotlib.pyplot as plt

history = train_result['history']
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
plots = [
    ('loss_total', 'Total Normalized Loss'),
    ('win_auc', 'Validation Win ROC-AUC'),
    ('dur_p50_mae', 'Duration p50 MAE (min)'),
    ('lane_acc', 'Lane Outcome Accuracy (Top/Mid/Bot)'),
    ('lead10_gold_mae', '@10m Gold Lead MAE'),
    ('hero_dpm_mae', 'Hero DPM MAE'),
    ('gpm_p50_mae', 'GPM p50 MAE'),
    ('item_acc', 'Item Next-Token Accuracy'),
    ('skill_acc', 'Skill Build Accuracy (1-18)'),
]
p2_epochs = [h['epoch'] for h in history if h.get('phase') == 2]
phase_split = min(p2_epochs) - 0.5 if p2_epochs else None
for ax, (key, title) in zip(axes.flat, plots, strict=True):
    tr_pts = [(h['epoch'], h['train'][key]) for h in history if key in h['train']]
    va_pts = [(h['epoch'], h['val'][key]) for h in history if key in h['val']]
    if tr_pts:
        ax.plot([p[0] for p in tr_pts], [p[1] for p in tr_pts], label='train')
    if va_pts:
        ax.plot([p[0] for p in va_pts], [p[1] for p in va_pts], label='val')
    if phase_split is not None:
        ax.axvline(phase_split, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 6. Export ONNX encoder + autoregressive item decoder and verify PyTorch vs ONNX parity
from training.export_onnx import export_to_onnx
from training.verify_onnx import verify_onnx_parity

exported = export_to_onnx(train_result['best_checkpoint'], cfg.export_dir)
parity = verify_onnx_parity(train_result['best_checkpoint'], cfg.export_dir, atol=1e-4)

total_mb = 0.0
for name, path in exported.items():
    size_mb = path.stat().st_size / (1024 * 1024)
    total_mb += size_mb
    print(f'{name}: {path} ({size_mb:.2f} MB)')
print(f'Total browser bundle size: {total_mb:.2f} MB')
print('ONNX parity:', parity)

In [ ]:
# 7. Inspect sample draft prediction via ONNX Runtime (auto vs 50-min scenario override)
import numpy as np
import onnxruntime as ort

from shared.constants import LANE_OUTCOME_LABELS
from training.dataset import BOS_ITEM_ID, EOS_ITEM_ID
from training.verify_onnx import decode_items_onnx_greedy

sample = train_result['sample_match']
hero_by_id = vocab['heroes']['by_id']
item_idx_to_id = vocab['items']['idx_to_id']
item_by_id = vocab['items']['by_id']

hero_idx_np = np.array([[vidx.hero_id_to_idx.get(int(h), 0) for h in sample['hero_ids']]], dtype=np.int64)
is_rad_np = np.array([[1 if f else 0 for f in sample['is_radiant']]], dtype=np.int64)
pos_idx_np = np.array([[int(p) - 1 for p in sample['positions']]], dtype=np.int64)

enc_sess = ort.InferenceSession(str(exported['encoder']), providers=['CPUExecutionProvider'])
dec_sess = ort.InferenceSession(str(exported['decoder']), providers=['CPUExecutionProvider'])

auto_out = enc_sess.run(
    None,
    {
        'hero_idx': hero_idx_np,
        'is_radiant': is_rad_np,
        'pos_idx': pos_idx_np,
        'duration_override': np.array([0.0], dtype=np.float32),
    },
)
win_prob, dur_q, lane_p, lead10, combat_r, combat_s, econ_q, skill_l, tal_l, slot_mem = auto_out
pred_ids, pred_mins = decode_items_onnx_greedy(
    dec_sess,
    slot_mem,
    max_steps=cfg.max_item_len - 1,
    bos_id=BOS_ITEM_ID,
    eos_id=EOS_ITEM_ID,
    eos_penalty=cfg.eos_logit_penalty,
    eos_stop_prob=cfg.eos_stop_prob,
)

long_out = enc_sess.run(
    None,
    {
        'hero_idx': hero_idx_np,
        'is_radiant': is_rad_np,
        'pos_idx': pos_idx_np,
        'duration_override': np.array([50.0], dtype=np.float32),
    },
)
_, _, _, _, long_combat_r, _, long_econ_q, _, _, long_slot_mem = long_out
long_ids, long_mins = decode_items_onnx_greedy(
    dec_sess,
    long_slot_mem,
    max_steps=cfg.max_item_len - 1,
    bos_id=BOS_ITEM_ID,
    eos_id=EOS_ITEM_ID,
    eos_penalty=cfg.eos_logit_penalty,
    eos_stop_prob=cfg.eos_stop_prob,
    max_minute=50.0,
)

print(f"Match {sample['match_id']} | Actual Radiant Win: {sample['radiant_win']} | Actual Duration: {sample['duration_seconds']/60:.1f} min")
print(f"Pred Radiant Win: {win_prob[0, 0]:.1%} | Duration: {dur_q[0, 0]:.1f}-{dur_q[0, 2]:.1f}m (~{dur_q[0, 1]:.1f}m)")
lane_str = ', '.join(
    f"{nm}={LANE_OUTCOME_LABELS[int(np.argmax(lane_p[0, i]))]} ({np.max(lane_p[0, i]):.0%})"
    for i, nm in enumerate(('Top', 'Mid', 'Bot'))
)
print(f"Lanes: {lane_str} | @10m Leads: Gold={lead10[0, 0]:+.0f}, XP={lead10[0, 1]:+.0f}\n")

for s in range(10):
    hid = int(sample['hero_ids'][s])
    hmeta = hero_by_id.get(str(hid), {})
    team = 'Radiant' if sample['is_radiant'][s] else 'Dire'
    pos = sample['positions'][s]
    t_idx = 0 if s < 5 else 1
    slot_t = s % 5
    items_fmt = []
    for tok, m in zip(pred_ids[0, s], pred_mins[0, s], strict=True):
        if int(tok) == EOS_ITEM_ID:
            break
        if int(tok) >= 3:
            iid = item_idx_to_id[int(tok)]
            iname = item_by_id.get(str(iid), {}).get('display_name', str(iid))
            items_fmt.append(f'{iname} (~{m:.0f}m)')
    gpm_rng = f"{econ_q[0, s, 0, 0]:.0f}-{econ_q[0, s, 0, 2]:.0f}"
    gpm_50 = f"{long_econ_q[0, s, 0, 1]:.0f}"
    hdpm = f"{combat_r[0, s, 0]:.0f} ({combat_s[0, t_idx, slot_t, 0]:.0%})"
    print(f"{team:7s} P{pos} {hmeta.get('display_name', hid):16s} | GPM:{gpm_rng} (50m:{gpm_50}) | HDPM:{hdpm} | Items: {', '.join(items_fmt) or 'none'}")

In [ ]:
# 8. Zip exported bundle for download or optional HF Hub upload (no hardcoded tokens)
import shutil

from huggingface_hub import HfApi

archive_path = shutil.make_archive('d2oracle_exports', 'zip', root_dir=cfg.export_dir)
print('Created bundle:', archive_path)

UPLOAD_REPO_ID = ''  # Optional: e.g. 'rizeru/d2oracle-model'
if UPLOAD_REPO_ID:
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Set HF_TOKEN in Colab Secrets or env before uploading.')
    api = HfApi(token=token)
    api.create_repo(repo_id=UPLOAD_REPO_ID, repo_type='model', exist_ok=True)
    api.upload_folder(folder_path=str(cfg.export_dir), repo_id=UPLOAD_REPO_ID, repo_type='model')
    print('Uploaded ONNX bundle to', UPLOAD_REPO_ID)